# SHIPP — Controlled DELETE Validation: Request

Purpose: validate real Lakebase → Bronze → Silver hard-delete semantics.

**Test ID:** `delete-test-request-001`

Important:
- Lakebase SQL below must be run in the **Lakebase SQL editor**.
- Do not delete demo records.
- Do not mark DELETE as VALIDATED until all gates pass.


## Gate 1 — Configuration


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "bootcamp_students.shipp_bronze.lb_requests_history"
TARGET_TABLE = "bootcamp_students.shipp_silver.silver_requests"
BUSINESS_KEY = "request_id"
DELETE_TEST_ID = "delete-test-request-001"

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("DELETE_TEST_ID =", DELETE_TEST_ID)


## Gate 2 — Capture current Silver baseline


In [ ]:
baseline_silver_ids = {
    row[BUSINESS_KEY]
    for row in spark.table(TARGET_TABLE).select(BUSINESS_KEY).collect()
}

print("Baseline Silver count:", len(baseline_silver_ids))
print("Baseline Silver IDs:", sorted(baseline_silver_ids))

assert DELETE_TEST_ID not in baseline_silver_ids, (
    "STOP: delete test ID already exists in Silver."
)

print("PASS — Test ID is not present in baseline Silver.")


## Gate 3 — Create temporary request in Lakebase

Run this in the **Lakebase SQL editor**:

```sql
SELECT user_id FROM shipp.users WHERE user_id = 'demo-requester-001';

INSERT INTO shipp.requests (
  request_id, requester_id, request_text, category, location,
  latitude, longitude, need_by_date, status
)
VALUES (
  'delete-test-request-001', 'demo-requester-001',
  'Temporary request created only to validate CDC delete semantics.',
  'FURNITURE', 'Abu Dhabi', 24.5014, 54.3872,
  CURRENT_TIMESTAMP + INTERVAL '7 days', 'OPEN'
);

SELECT request_id, requester_id, request_text, status
FROM shipp.requests
WHERE request_id = 'delete-test-request-001';
```

Expected:
- parent user query returns one row
- test row is inserted
- final SELECT returns exactly one `delete-test-request-001` row


## Gate 4 — Confirm INSERT reached Bronze


In [ ]:
bronze_df = spark.table(SOURCE_TABLE)

test_history_df = (
    bronze_df
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .select(
        BUSINESS_KEY,
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status", "updated_at"
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc()
    )
)

display(test_history_df)

insert_count = (
    test_history_df
    .filter(F.col("_pg_change_type") == "insert")
    .count()
)

print("Observed INSERT events:", insert_count)

assert insert_count >= 1, (
    "STOP: Bronze INSERT not observed yet. Wait for CDF and rerun."
)

print("PASS — Bronze contains the temporary INSERT event.")


## Gate 5 — Confirm temporary row reaches Silver

Run `11_silver_requests_dev.ipynb` once with the current Stage 5 logic.

Then run the next cell.


In [ ]:
before_delete_count = (
    spark.table(TARGET_TABLE)
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .count()
)

print("Temporary entity in Silver before delete:", before_delete_count)

assert before_delete_count == 1, (
    "STOP: temporary entity did not reach Silver before delete."
)

print("PASS — Temporary entity exists in Silver before delete.")


## Gate 6 — Delete the temporary request in Lakebase

Run this in the **Lakebase SQL editor**:

```sql
DELETE FROM shipp.requests
WHERE request_id = 'delete-test-request-001'
RETURNING request_id;

SELECT *
FROM shipp.requests
WHERE request_id = 'delete-test-request-001';
```

Expected:
- DELETE returns `delete-test-request-001`
- final SELECT returns **0 rows**


## Gate 6A — Inspect the real Bronze DELETE event


In [ ]:
bronze_df = spark.table(SOURCE_TABLE)

delete_history_df = (
    bronze_df
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .select(
        BUSINESS_KEY,
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status", "updated_at"
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc()
    )
)

display(delete_history_df)

assert delete_history_df.count() > 0

latest_test_event = delete_history_df.first()

print("Latest change type:", latest_test_event["_pg_change_type"])
print("Latest LSN:", latest_test_event["_pg_lsn"])
print("Latest sort:", latest_test_event["_sort_by"])

assert latest_test_event["_pg_change_type"] == "delete", (
    "STOP: newest Bronze event is not delete. "
    "Do not patch Silver until runtime evidence confirms it."
)

print("PASS — Newest Bronze event is a real DELETE.")


## Gate 7 — Apply delete-aware reconstruction in `11_silver_requests_dev.ipynb`

Only after Gate 6A passes, update the Stage 5 notebook to this pattern:

```python
CURRENT_STATE_CHANGE_TYPES = {
    "insert",
    "update_postimage",
    "delete",
}

VALID_SILVER_SOURCE_CHANGE_TYPES = {
    "insert",
    "update_postimage",
}

current_state_events_df = (
    bronze_df
    .filter(
        F.col("_pg_change_type").isin(
            list(CURRENT_STATE_CHANGE_TYPES)
        )
    )
)

latest_window = (
    Window.partitionBy(BUSINESS_KEY)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

ranked_df = (
    current_state_events_df
    .withColumn(
        "_current_rn",
        F.row_number().over(latest_window)
    )
)

latest_events_df = ranked_df.filter(F.col("_current_rn") == 1)

deleted_latest_df = latest_events_df.filter(
    F.col("_pg_change_type") == "delete"
)

current_rows_df = latest_events_df.filter(
    F.col("_pg_change_type") != "delete"
)
```

Then build `silver_df` from `current_rows_df`, not `latest_events_df`.


## Gate 8 — Run patched `11_silver_requests_dev.ipynb`

Run it from top to bottom, then return here.


In [ ]:
deleted_test_count = (
    spark.table(TARGET_TABLE)
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .count()
)

print("Deleted test entity remaining in Silver:", deleted_test_count)

assert deleted_test_count == 0, (
    "DELETE VALIDATION FAILED: deleted test entity still exists in Silver."
)

print("PASS — Deleted test entity is absent from Silver.")


## Gate 9 — Prove unrelated Silver rows are unchanged


In [ ]:
final_silver_ids = {
    row[BUSINESS_KEY]
    for row in spark.table(TARGET_TABLE).select(BUSINESS_KEY).collect()
}

print("Baseline IDs:", sorted(baseline_silver_ids))
print("Final IDs:   ", sorted(final_silver_ids))

assert final_silver_ids == baseline_silver_ids, (
    "DELETE VALIDATION FAILED: unrelated Silver entities changed."
)

print("PASS — Unrelated Silver entities remain unchanged.")


## Gate 10 — Prove every latest DELETE is absent from Silver


In [ ]:
current_state_events_df = (
    spark.table(SOURCE_TABLE)
    .filter(
        F.col("_pg_change_type").isin(
            "insert", "update_postimage", "delete"
        )
    )
)

latest_window = (
    Window.partitionBy(BUSINESS_KEY)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc()
    )
)

latest_events_df = (
    current_state_events_df
    .withColumn("_current_rn", F.row_number().over(latest_window))
    .filter(F.col("_current_rn") == 1)
)

deleted_latest_ids_df = (
    latest_events_df
    .filter(F.col("_pg_change_type") == "delete")
    .select(BUSINESS_KEY)
    .distinct()
)

deleted_rows_still_in_silver = (
    spark.table(TARGET_TABLE)
    .join(deleted_latest_ids_df, on=BUSINESS_KEY, how="inner")
    .count()
)

print(
    "Latest-delete entities still present in Silver:",
    deleted_rows_still_in_silver
)

assert deleted_rows_still_in_silver == 0

print("PASS — Latest DELETE entities are excluded from Silver.")


## Gate 11 — Idempotency

Capture Run 1 below.  
Then rerun `11_silver_requests_dev.ipynb` one more time **without changing Lakebase data**.  
Then run the Run 2 comparison cell.


In [ ]:
logical_columns = [
    col
    for col in spark.table(TARGET_TABLE).columns
    if col != "silver_processed_at"
]

delete_run1_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

print("DELETE validation Run 1 rows:", len(delete_run1_rows))
print("PASS — Run 1 logical state captured.")


**Now rerun `11_silver_requests_dev.ipynb` once, then run the next cell.**


In [ ]:
delete_run2_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

print("DELETE validation Run 1 rows:", len(delete_run1_rows))
print("DELETE validation Run 2 rows:", len(delete_run2_rows))

assert delete_run1_rows == delete_run2_rows, (
    "IDEMPOTENCY FAILED: logical Silver state changed after rerun."
)

assert DELETE_TEST_ID not in {
    row[BUSINESS_KEY] for row in delete_run2_rows
}

print("PASS — DELETE-aware Silver rerun is logically idempotent.")


## Final acceptance

Mark:

```text
REQUEST INSERT  → VALIDATED
REQUEST UPDATE  → VALIDATED
REQUEST DELETE  → VALIDATED
```

only when:
- temporary row existed before delete
- Bronze captured a real `delete`
- delete was newest by `_pg_lsn`, `_sort_by`
- deleted row is absent from Silver
- unrelated Silver rows are unchanged
- latest-delete entities are absent from Silver
- rerun is logically idempotent
